# Gemma 4 31B FBSFM fine-tune - v2 (prompt-mismatch fix)

This notebook **re-trains the LoRA adapter so that the training format is exactly the format used at extraction time.**

## What was wrong with v1

| | v1 training (`gemma_training_colab.ipynb`) | Extraction / validation (`colab_other_extract.py`, cell 16) |
|---|---|---|
| User prompt | short: *"Below is an OCR page from a biology textbook. Extract the knowledge graph in JSON format. ### Page Text: ..."* | the long **Ed-Machine FBSFM prompt** (naming rules, schema, relationship verbs, ~1.8k tokens) |
| Start of the model turn | `<|turn>model\n` (role `model` messages render **without** a thought channel) | `<|turn>model\n<|channel>thought\n<channel|>` (Gemma 4's template appends this when thinking is off) |
| Reply | `### Extracted JSON:\n{...}` | bare JSON expected |
| Loss | on the **whole** sequence, including the OCR text | - |
| Target keys | `_page`, alphabetical order | schema asks for `page`, `name` first |

So at extraction time the model was asked a question it had **never seen in training**, inside a turn prefix it had never seen. That is what produced the "responded like a conversation" output, the missing `### Extracted JSON:` header, and the drift into off-page entities (it was falling back on the base model's behaviour).

## What v2 changes

1. **Same prompt**: every training example is built with the *exact* production template (verified below against `other_resources_in` - it rebuilds the production prompts byte-for-byte).
2. **Same turn prefix**: the training text is `apply_chat_template([user], add_generation_prompt=True)` + target JSON + `<turn|>`, i.e. literally what the model sees at inference, plus the answer.
3. **Bare JSON reply** in schema key order (`page` not `_page`), Devanagari kept as real characters (not JSON unicode escapes, which cost several tokens per character).
4. **Loss only on the reply** (`train_on_responses_only`), checked by a cell that decodes what is actually trained.
5. **No silent truncation**: examples longer than `MAX_SEQ` are dropped (and counted) instead of being cut mid-JSON.
6. **Held-out split** (1%) saved to Drive, so validation is on pages the model did not train on.
7. **Separate output folders** (`outputs_v2/`, `gemma_fbsfm_lora_v2/`) - v1's `outputs/` already holds the finished `checkpoint-3319`, so "resume" there would do nothing.
8. Writes `prompt_style.txt = v2` into the adapter folder; `validate_gemma_lora.py` and `colab_other_extract_v4.py` read it and prompt accordingly.

**Cost:** each example now carries the ~1.8k-token instructions, so expect roughly 1.5-2x the v1 training time on the A100.

> Cheaper alternative (no retraining): keep the v1 adapter and run extraction with `--prompt-style v1`, which sends the short training prompt. The v2 route is the clean fix if you want the model to follow the full naming/schema rules it is given.

## 1. Install

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

## 2. Mount Drive and configure

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
ROOT        = "/content/drive/MyDrive/gemma_ft"
DATASET     = f"{ROOT}/gemma_training_dataset.jsonl.gz"
CHUNK_DIR   = f"{ROOT}/other_resources_in"          # used only to verify the prompt template
OUTPUT_DIR  = f"{ROOT}/outputs_v2"                  # NEW folder - do not reuse v1's outputs/
ADAPTER_DIR = f"{ROOT}/gemma_fbsfm_lora_v2"         # NEW folder - v1 adapter stays untouched
HELDOUT     = f"{ROOT}/heldout_v2.jsonl.gz"

BASE_MODEL  = "unsloth/gemma-4-31B-it-unsloth-bnb-4bit"
MAX_SEQ     = 8192     # prompt (~1.8k tokens + page) + JSON reply; longer examples are dropped
HELDOUT_FRAC = 0.01
SEED        = 3407
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Look at the training rows

The production prompt needs the **page number** and the **PDF name**. This cell shows the fields of the first rows and how `row_page` / `row_pdf` resolve them. If the PDF name comes out empty, add the right key name to `PDF_KEYS`.

In [ ]:
import gzip, json, itertools

PAGE_KEYS = ("page", "page_num", "page_number", "_page")
PDF_KEYS  = ("pdf_file", "pdf_name", "source_pdf", "pdf", "source", "book")

def _graph(row):
    try:
        g = json.loads(row["extracted_graph"])
        return g if isinstance(g, dict) else {}
    except Exception:
        return {}

def row_page(row):
    for k in PAGE_KEYS:
        if row.get(k) not in (None, ""):
            return int(row[k])
    g = _graph(row)
    if g.get("page") not in (None, ""):
        return int(g["page"])
    for e in g.get("entities", []):
        if isinstance(e, dict) and e.get("_page", e.get("page")) is not None:
            return int(e.get("_page", e.get("page")))
    return None

def row_pdf(row):
    for k in PDF_KEYS:
        if row.get(k):
            return os.path.basename(str(row[k]))
    return _graph(row).get("pdf_file") or ""

with gzip.open(DATASET, "rt", encoding="utf-8") as f:
    sample = [json.loads(l) for l in itertools.islice(f, 2000)]

print("Fields:", list(sample[0].keys()))
for r in sample[:3]:
    print({k: str(v)[:70] for k, v in r.items()})
print()
print("page resolved :", sum(row_page(r) is not None for r in sample), "/", len(sample))
print("pdf  resolved :", sum(bool(row_pdf(r)) for r in sample), "/", len(sample))
print("example ->", row_page(sample[0]), repr(row_pdf(sample[0])))

## 4. The production prompt template

Copied verbatim from the `prompt` field of the extraction chunks. The check below rebuilds the prompts of a chunk file from (page, pdf, page text) and must report a 100% match - that is the guarantee that training and extraction see the same prompt.

In [ ]:
PROMPT_TEMPLATE = """You are an Ed-Machine knowledge graph builder for multilingual (Hindi + English) textbooks.

Focus ONLY on PAGE {page} of source document "{pdf_name}".
The OCR'd content of that page is given below under =====PAGE TEXT=====.

Extract with MAXIMUM density. Do NOT skip anything. There is NO target, minimum or maximum:
extract EVERY entity and EVERY relationship this page supports (a dense page may yield 100+).
If the page has no subject content (blank, cover, index-only), return empty arrays.

===== CRITICAL BILINGUAL NAMING RULES — read carefully =====

Every entity has THREE name fields with strict rules:

1. `name`  (canonical key, used for Neo4j lookups)
   - MUST be the ENGLISH scientific/common term in UPPERCASE_SNAKE_CASE.
   - If the English term exists (which it almost always does for science/biology/physics/chemistry/math), use it — even if the source page is in Hindi.
   - Example: Hindi page mentions "आनुवंशिकता" -> name="HEREDITY"
   - Example: Hindi page mentions "प्रकाश संश्लेषण" -> name="PHOTOSYNTHESIS"

2. `name_en` (human-readable English) — proper Title Case English name.

3. `name_hi` (Devanagari script — never Romanized)
   - Use the EXACT Devanagari as it appears, or the standard Devanagari translation if only English source is present.
   - If the concept has no Hindi equivalent (e.g. a proper noun like "DNA"), set name_hi to "".

4. NO positional or context-only names. Names are merged across ALL books in Neo4j, so
   FIRST_PROTEASE, SECOND_ENZYME, TARGET_GENES, R1_SUBSTITUENT, THIS_PATHWAY would wrongly merge
   unrelated things. This applies EVEN WHEN the page itself only says "first protease" or
   "target genes" (common in diagram labels): the name MUST NEVER start with FIRST_, SECOND_,
   THIRD_, THIS_, THAT_, TARGET_, R1_/R2_. If you are certain of the standard name, use it
   (SITE_1_PROTEASE, SITE_2_PROTEASE) and put the page's wording in description; otherwise
   prefix the context (SREBP_FIRST_PROTEASE, SREBP_TARGET_GENES, STEROID_R1_SUBSTITUENT).

===== FORBIDDEN: HINGLISH / ROMANIZED HINDI =====

NEVER output Romanized/Hinglish transliterations in ANY field:
   WRONG "AANUVANSHIKTA"       -> use "HEREDITY" + name_hi="आनुवंशिकता"
   WRONG "PRAKASH_SANSHLESHAN" -> use "PHOTOSYNTHESIS" + name_hi="प्रकाश संश्लेषण"

===== SCHEMA =====

Return ONLY a JSON object (no prose, no markdown fences):

{
  "page": {page},
  "pdf_file": "{pdf_name}",
  "entities": [
    {
      "name": "ENGLISH_CANONICAL_UPPERCASE_NAME",
      "name_en": "Human-readable English title",
      "name_hi": "देवनागरी लिपि में",
      "type": "Structure|Process|Concept|Principle|Condition|FailureMode|Chemical|Molecule|Protein|Enzyme|Receptor|Hormone|Pathway|Organelle|Formula|Law|Person|Species|Location|Technique|Disease|Misconception|Device|Organ|Cell|Tissue|Gene|Quantity|Property|Event|Figure|Table",
      "fbsfm_class": "S|B|F|P|C|M",
      "description": "1-2 sentences from page {page} content",
      "page": {page}
    }
  ],
  "relationships": [
    {
      "source": "EXACT entity name from entities array above",
      "target": "EXACT entity name from entities array above",
      "type": "IS_A|PART_OF|COMPONENT_OF|CONTAINS|SUBUNIT_OF|MEMBER_OF|CAUSES|LEADS_TO|PRODUCES|REQUIRES|ENABLES|PREVENTS|PRECEDES|ACTIVATES|INHIBITS|REGULATES|UPREGULATES|DOWNREGULATES|CONTROLS|BINDS_TO|CATALYZES|PHOSPHORYLATES|CLEAVES|TRANSPORTS|RECEPTOR_FOR|ENCODES|TRANSCRIBED_TO|TRANSLATED_TO|EXPRESSED_IN|REACTS_WITH|OXIDIZES_TO|REDUCES_TO|DECOMPOSES_TO|DISSOLVES_IN|DISPLACES|CONVERTED_TO|SUBSTRATE_OF|COFACTOR_OF|ALLELE_OF|DOMINANT_OVER|EPISTATIC_TO|LINKED_TO|MUTANT_OF|INHERITED_VIA|DIFFERENTIATES_INTO|INDUCES|DERIVED_FROM|SECRETED_BY|LOCATED_IN|RECOGNIZES|INFECTS|CAUSATIVE_AGENT_OF|SYMPTOM_OF|EVOLVED_FROM|HOMOLOGOUS_TO|ANALOGOUS_TO|ENDEMIC_TO|PREYS_ON|PARASITIZES|MUTUALISTIC_WITH|COMPETES_WITH|HABITAT_OF|MEASURES|USED_IN|APPLIED_IN|DETECTS|SEPARATES_BY|BASED_ON|DISCOVERED|PROPOSED_BY|DEFINED_BY|OFTEN_CONFUSED_WITH|MISCONCEPTION_OF|PREREQUISITE_OF|CONTRASTS_WITH|SYNONYM_OF|ALSO_KNOWN_AS|DEPICTED_IN|ILLUSTRATED_BY|LABELED_IN|LISTED_IN|COLUMN_OF|COMPARES|...or any other specific UPPERCASE_SNAKE_CASE verb",
      "description": "short phrase explaining why this relationship holds",
      "page": {page}
    }
  ]
}

Extract EVERY named thing on this page across all six FBSFM classes:
  S (Structural) — organs, cells, tissues, chemicals, apparatus, species, locations, materials
  B (Behavioral) — processes, reactions, techniques, procedures, experiments, actions
  F (Functional) — concepts, properties, quantities, formulas, traits, functions
  P (Principle)  — laws, theorems, theories, rules, axioms
  C (Condition)  — states, phases, environmental conditions, constraints
  M (Failure)    — misconceptions, disorders, diseases, common errors, mutations

Prefer highly specific verbs (e.g., PHOSPHORYLATES, COMPETITIVELY_INHIBITS) over generic ones (RELATED_TO). If you encounter a specific biological/chemical interaction not in the main list, create a precise UPPERCASE_SNAKE_CASE verb for it on the spot.
Write every relationship as "source VERB target" in the ACTIVE direction (ENZYME CATALYZES REACTION,
not REACTION CATALYZED_BY ENZYME).

Where the page contrasts, compares, or warns about two things (C3 vs C4, competitive vs
non-competitive inhibition), add OFTEN_CONFUSED_WITH / CONTRASTS_WITH between them, and a
Misconception entity with MISCONCEPTION_OF when the page names a common error. Stay grounded
in the page - do not invent confusions it does not support.

Figures/tables: lines marked [FIGURE] or under FIGURE CAPTIONS come from diagrams. Emit one Figure
entity FIGURE_<TOPIC>_<N> (or Table entity TABLE_<TOPIC>_<N>); labelled parts -> PART_OF/LABELED_IN,
depicted processes -> DEPICTED_IN, row subjects -> LISTED_IN.
Output ONLY the JSON. No markdown fences. No explanatory text before or after.

Relationship types already coined in this corpus (reuse one when it fits instead of inventing a near-duplicate): DEPICTS, INCLUDES, UTILIZES, EXHIBITS, LABELS, INVOLVES, ACTS_ON, OCCURS_IN, HAS_PROPERTY, COMPOSED_OF, LISTS, ILLUSTRATES, CONSUMES, UNDERGOES, PARTICIPATES_IN, DEPENDS_ON, INCREASES, APPLIED_TO, MEDIATES, INFLUENCES, DETERMINES, REDUCES, APPLIES_TO, CALCULATED_FROM, FORMS, DESCRIBES, TRANSPORTS_TO, AUTHORED, USES, PROMOTES, DRIVES, OCCURS_AT, PROPERTY_OF, PROVIDES, ACTS_AS, CONTRIBUTES_TO, CALCULATES, GENERATES, MODIFIES, RELEASES, ATTACHED_TO, IDENTIFIES, INTERACTS_WITH, DEFINES, EXPRESSES, PLOTS, ASSOCIATES_WITH, SECRETES, PROPOSED, COVERS, PRECURSOR_OF, STIMULATES, CHARACTERIZES, EXPLAINS, RELATES, CONSISTS_OF, REPRESENTS, STABILIZES, AFFECTS, REVIEWED

=====PAGE TEXT=====
{page_text}
=====END PAGE TEXT=====
"""

import glob, re

def build_prompt(page, pdf_name, page_text):
    # .replace, not .format: the template contains literal JSON braces
    p = PROMPT_TEMPLATE.replace("{page}", str(page)).replace("{pdf_name}", pdf_name)
    return p.replace("{page_text}", page_text)

chunks = sorted(glob.glob(f"{CHUNK_DIR}/other_resources_chunk_*.jsonl.gz"))
if chunks:
    with gzip.open(chunks[0], "rt", encoding="utf-8", errors="replace") as f:
        rows = [json.loads(l) for l in itertools.islice(f, 500) if l.strip()]
    same = 0
    for r in rows:
        m = re.search(r"=====PAGE TEXT=====\n(.*?)\n=====END PAGE TEXT=====", r["prompt"], re.S)
        same += bool(m) and build_prompt(r["page"], r["pdf_name"], m.group(1)) == r["prompt"]
    print(f"Template rebuilds {same}/{len(rows)} production prompts exactly")
    assert same >= 0.95 * len(rows), "production prompt changed - update PROMPT_TEMPLATE before training"
else:
    print("No chunk files found - skipping the template check")

## 5. Load the base model and add fresh LoRA adapters

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ,
    dtype = None,
    load_in_4bit = True,
    text_only = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)
tok = getattr(tokenizer, "tokenizer", tokenizer)

GEN_PREFIX_END = "<|channel>thought\n<channel|>"
probe = tok.apply_chat_template([{"role": "user", "content": "hi"}], tokenize=False, add_generation_prompt=True)
print(repr(probe))
assert probe.endswith(GEN_PREFIX_END), "chat template generation prompt is not what this notebook expects"

## 6. Build the training text

`text = <chat-template generation prompt for the production prompt> + <target JSON> + "<turn|>"`

In [ ]:
from datasets import load_dataset

ENT_ORDER = ["name", "name_en", "name_hi", "type", "fbsfm_class", "description", "page"]
REL_ORDER = ["source", "target", "type", "description", "page"]

def _ordered(item, order, page):
    item = dict(item)
    p = item.pop("_page", None)
    item.setdefault("page", p if p is not None else page)
    out = {k: item[k] for k in order if k in item}
    out.update({k: v for k, v in item.items() if k not in out and not k.startswith("_")})
    return out

def normalize_target(graph, page, pdf):
    return json.dumps({
        "page": page,
        "pdf_file": pdf,
        "entities": [_ordered(e, ENT_ORDER, page) for e in graph.get("entities", []) if isinstance(e, dict)],
        "relationships": [_ordered(r, REL_ORDER, page) for r in graph.get("relationships", []) if isinstance(r, dict)],
    }, ensure_ascii=False)

def make_examples(batch):
    out = {"text": [], "prompt": [], "n_tokens": [], "page": [], "pdf_name": []}
    for i in range(len(batch["raw_ocr_text"])):
        row = {k: batch[k][i] for k in batch}
        page, pdf = row_page(row), row_pdf(row)
        graph = _graph(row)
        if page is None or not isinstance(graph.get("entities"), list):
            text, prompt, n = "", "", 10**9          # unusable row -> filtered out below
        else:
            prompt = build_prompt(page, pdf, row["raw_ocr_text"])
            head = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                           tokenize=False, add_generation_prompt=True)
            text = head + normalize_target(graph, page, pdf) + "<turn|>"
            n = len(tok(text, add_special_tokens=False)["input_ids"])
        out["text"].append(text); out["prompt"].append(prompt); out["n_tokens"].append(n)
        out["page"].append(page if page is not None else -1); out["pdf_name"].append(pdf)
    return out

raw = load_dataset("json", data_files=DATASET, split="train")
ds = raw.map(make_examples, batched=True, batch_size=256, num_proc=4)
before = len(ds)
ds = ds.filter(lambda n: n <= MAX_SEQ, input_columns="n_tokens")
print(f"Kept {len(ds)} / {before} examples ({before - len(ds)} dropped: too long or no page/graph)")
print("tokens per example: median", sorted(ds["n_tokens"])[len(ds)//2], " max", max(ds["n_tokens"]))

split = ds.train_test_split(test_size=HELDOUT_FRAC, seed=SEED)
train_ds, heldout_ds = split["train"], split["test"]

# Held-out pages for validate_gemma_lora.py --prompt v2 (rows keep `prompt` + ground truth)
with gzip.open(HELDOUT, "wt", encoding="utf-8") as f:
    for r in heldout_ds:
        f.write(json.dumps({"page": r["page"], "pdf_name": r["pdf_name"], "raw_ocr_text": r["raw_ocr_text"],
                            "extracted_graph": r["extracted_graph"], "prompt": r["prompt"]},
                           ensure_ascii=False) + "\n")
print(f"train: {len(train_ds)}   held-out: {len(heldout_ds)} -> {HELDOUT}")
print("\n--- end of one training text ---\n", train_ds[0]["text"][-700:])

## 7. Trainer - loss on the reply only

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds.select_columns(["text"]),
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,      # longer sequences than v1 -> smaller batch,
        gradient_accumulation_steps = 8,      # same effective batch of 16
        warmup_steps = 20,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = SEED,
        output_dir = OUTPUT_DIR,
        save_strategy = "steps",
        save_steps = 50,
        save_total_limit = 2,
        report_to = "none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n" + GEN_PREFIX_END,
)

## 8. Check what is actually trained

Must print a trained part that **starts with `{"page":`** and ends with `<turn|>`. If the assert fails, the masking markers do not match the template - do not start training.

In [ ]:
item = trainer.train_dataset[0]
if "labels" in item:
    ids, labels = item["input_ids"], item["labels"]
else:
    batch = trainer.data_collator([{k: item[k] for k in ("input_ids", "attention_mask") if k in item}])
    ids, labels = batch["input_ids"][0].tolist(), batch["labels"][0].tolist()

trained = [t for t, l in zip(ids, labels) if l != -100]
txt = tok.decode(trained)
print(f"masked (prompt) tokens: {sum(l == -100 for l in labels)}   trained (reply) tokens: {len(trained)}")
print("trained part starts:", repr(txt[:120]))
print("trained part ends  :", repr(txt[-60:]))
assert txt.lstrip().startswith('{"page"'), "loss is not restricted to the JSON reply"

## 9. Train (resumes automatically if `outputs_v2/` already has a checkpoint)

In [ ]:
import glob
ckpts = glob.glob(f"{OUTPUT_DIR}/checkpoint-*")
print("Resuming from latest checkpoint" if ckpts else "Starting fresh")
trainer_stats = trainer.train(resume_from_checkpoint = bool(ckpts))

## 10. Save the v2 adapter

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
with open(f"{ADAPTER_DIR}/prompt_style.txt", "w") as f:
    f.write("v2")
print("Saved", ADAPTER_DIR, os.listdir(ADAPTER_DIR))

## 11. Validate on held-out pages

**First: Runtime -> Restart session** (keeps the installed packages, frees the GPU - the scripts start their own process and need the memory). Then re-run only cell 2 (mount/config) and the cells below.

In [ ]:
# held-out pages: ground truth available -> P/R/F1 plus schema/grounding checks
!python {ROOT}/validate_gemma_lora.py --adapter {ADAPTER_DIR} --data train --dataset {HELDOUT} --n 30 --report {ROOT}/validation_v2_heldout.json

# unseen production pages (other_resources_in): grounding/schema checks only
!python {ROOT}/validate_gemma_lora.py --adapter {ADAPTER_DIR} --data chunk --n 20 --report {ROOT}/validation_v2_chunk.json

## 12. Batch extraction with the v2 adapter (only once validation passes)

In [ ]:
!python -u {ROOT}/colab_other_extract_v4.py \
    --input {ROOT}/other_resources_in \
    --adapter {ADAPTER_DIR} \
    --out {ROOT}/other_resources_out_v2 \
    --batch 4 --max-new 8192